# `SchemaView.namespaces()` permanently caches an incomplete prefix map when called during its own import resolution

Prepared as supporting evidence for a new bug report filed directly against
[linkml/linkml](https://github.com/linkml/linkml) (a `linkml_runtime`-package
bug -- `linkml/linkml-runtime` is archived, development moved into this
monorepo's own `packages/linkml_runtime/`).

**Found while diagnosing** Health-DCAT-AP-plus's own `title`-and-friends
SHACL finding (see `docs/architecture-verification.md`'s "A tenth
finding" section) -- fixed there by adding a missing `xsd:` prefix
redeclaration, which resolved it in that specific schema. Re-investigated
after being asked directly whether that fix could be hiding a deeper
problem, since several hand-built minimal schemas failed to reproduce it
even though the mechanism looked like it should be simple. It wasn't --
here's the real one, isolated and proven.

**Related but distinct**: [linkml/linkml#3574](https://github.com/linkml/linkml/issues/3574)
("prefixes declared by sub-schema are dropped during (merge) import")
already documents two causes (a commented-out line in `SchemaLoader`'s
own `merge_namespaces`, and `SchemaView` never copying imported prefixes
onto `schema.prefixes`) and has an open fix,
[PR #3575](https://github.com/linkml/linkml/pull/3575). **Neither
mentions this mechanism**, and this notebook proves PR #3575's own fix
(`materialize_prefixes()`) does not resolve this specific case -- see the
last section.

**Zero dependency on dcat-ap-plus, Health-DCAT-AP-plus, or any external
schema** -- three tiny synthetic schemas, all written to a temp directory
by this notebook itself, so it's fully portable and self-contained.

In [1]:
import tempfile
import traceback
from functools import lru_cache
from pathlib import Path

from linkml.generators.shaclgen import ShaclGenerator
from linkml_runtime.utils.schemaview import SchemaView
from rdflib import Graph, URIRef

tmpdir = Path(tempfile.mkdtemp(prefix="linkml-namespaces-cache-bug-"))
# Path.as_uri() -- the whole point of this bug needs a genuine, loadable
# CURIE-style import (one that has to be resolved via SchemaView's own
# namespaces(), not a relative path, which bypasses namespaces() entirely
# -- see the third markdown cell below). A local file:// URI reproduces
# the exact same code path a real w3id-permalink CURIE import goes
# through, with no network dependency.
childns_uri = tmpdir.as_uri() + "/"
print(f"childns: {childns_uri}")

childns: file:///C:/Users/REMY~1.BEN/AppData/Local/Temp/linkml-namespaces-cache-bug-dxiuhwy9/


## 1. The three schemas

- **`child.yaml`** -- one class, one `range: string` slot. Nothing special.
- **`local_helper.yaml`** -- a schema `root.yaml` imports by relative path
  (`./local_helper`, no CURIE resolution needed to find it), which *also*
  independently imports `childns:child` -- the same CURIE `root.yaml`
  itself imports. This re-declaration is the essential ingredient; see
  section 2 for why.
- **`root.yaml`** -- imports `linkml:types`, `childns:child`, and
  `./local_helper`, in that order, and defines `RootThing`, `is_a Thing`
  (inheriting `child.yaml`'s own string-typed `label` slot).

In [2]:
(tmpdir / "child.yaml").write_text(
    """\
id: https://example.org/child
name: child
prefixes:
  linkml: https://w3id.org/linkml/
  child: https://example.org/child/
default_range: string
default_prefix: child
imports:
  - linkml:types

classes:
  Thing:
    class_uri: child:Thing
    attributes:
      label:
        range: string
""",
    encoding="utf-8",
)

(tmpdir / "local_helper.yaml").write_text(
    f"""\
id: https://example.org/local-helper
name: local-helper
prefixes:
  linkml: https://w3id.org/linkml/
  helper: https://example.org/helper/
  childns: {childns_uri}
default_range: string
default_prefix: helper
imports:
  - linkml:types
  - childns:child

classes:
  HelperThing:
    class_uri: helper:HelperThing
    attributes:
      note:
        range: string
""",
    encoding="utf-8",
)

(tmpdir / "root.yaml").write_text(
    f"""\
id: https://example.org/root
name: root
prefixes:
  linkml: https://w3id.org/linkml/
  root: https://example.org/root/
  childns: {childns_uri}
default_range: string
default_prefix: root
imports:
  - linkml:types
  - childns:child
  - ./local_helper

classes:
  RootThing:
    is_a: Thing
""",
    encoding="utf-8",
)
print(f"Wrote 3 schema files to {tmpdir}")

Wrote 3 schema files to C:\Users\REMY~1.BEN\AppData\Local\Temp\linkml-namespaces-cache-bug-dxiuhwy9


## 2. Trace exactly when `namespaces()` gets called, and what it can see

`SchemaView.namespaces()` is decorated `@lru_cache(None)` -- computed
once per `SchemaView` instance, cached forever, no invalidation when
`self.schema_map` later grows. `imports_closure()` (called by
`ShaclGenerator.as_graph()` via `sv.all_classes(imports=True)`) processes
imports off a **stack** (`todo.pop()` from the end) -- so imports are
visited in *reverse* file-declaration order, recursively. `root.yaml`
declares `[linkml:types, childns:child, ./local_helper]`; the stack pops
`./local_helper` first, which -- because it *also* imports `childns:child`
-- pushes that CURIE fresh onto the stack, ahead of `root.yaml`'s own
still-pending `linkml:types`. Resolving `childns:child` (a real CURIE,
unlike `linkml:`, which is bootstrapped for free via a hardcoded
`{"linkml:": SCHEMA_DIRECTORY}` map and never touches `namespaces()` at
all) needs `self.namespaces()` to expand the prefix -- at a moment when
`linkml:types` (the schema that actually declares `xsd:`) has not been
loaded yet.

In [3]:
orig_namespaces = SchemaView.namespaces.__wrapped__
call_count = [0]


def traced_namespaces(self):
    call_count[0] += 1
    print(f"=== namespaces() actually computing (call #{call_count[0]}) ===")
    print(f"    schema_map at this moment: {list(self.schema_map.keys())}")
    print(f"    xsd already present? {'xsd' in orig_namespaces(self)}")
    return orig_namespaces(self)


SchemaView.namespaces = lru_cache(None)(traced_namespaces)

gen = ShaclGenerator(str(tmpdir / "root.yaml"))
shapes_ttl = gen.serialize()

=== namespaces() actually computing (call #1) ===
    schema_map at this moment: ['root', './local_helper']
    xsd already present? False


`namespaces()` computes exactly once, with `schema_map` containing only
`root` and `./local_helper` -- `linkml:types` isn't in there yet. That
single, incomplete result is what every later caller gets for the rest
of this `SchemaView`'s lifetime.

## 3. The result: a malformed `sh:datatype`

In [4]:
print(shapes_ttl)

g = Graph()
g.parse(data=shapes_ttl, format="turtle")
SH_DATATYPE = URIRef("http://www.w3.org/ns/shacl#datatype")
for s, p, o in g:
    if p == SH_DATATYPE:
        print(f"sh:datatype term: {o!r}  (length {len(str(o))})")

@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix root: <https://example.org/root/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<child:Thing> a sh:NodeShape ;
    sh:closed true ;
    sh:ignoredProperties ( rdf:type ) ;
    sh:property [ sh:datatype <xsd:string> ;
            sh:maxCount 1 ;
            sh:nodeKind sh:Literal ;
            sh:order 0 ;
            sh:path <child:label> ] ;
    sh:targetClass <child:Thing> .

<https://example.org/helper/HelperThing> a sh:NodeShape ;
    sh:closed true ;
    sh:ignoredProperties ( rdf:type ) ;
    sh:property [ sh:datatype <xsd:string> ;
            sh:maxCount 1 ;
            sh:nodeKind sh:Literal ;
            sh:order 0 ;
            sh:path <https://example.org/helper/note> ] ;
    sh:targetClass <https://example.org/helper/HelperThing> .

root:RootThing a sh:NodeShape ;
    sh:closed true ;
    sh:ignoredProperties ( rdf:type ) ;
    sh:property [ sh:datatype 

`xsd:string` -- the ten-character, unexpanded CURIE string, used directly
as a `URIRef` -- not the real, 39-character
`http://www.w3.org/2001/XMLSchema#string`. No literal, however typed,
could ever satisfy that `sh:datatype` constraint.

## 4. Proof it's a caching bug, not a resolution bug

Manually clearing the cache and re-resolving, on the *same* `SchemaView`
instance, after the exact same import closure has already finished
loading, fixes it immediately -- confirming the underlying data
(`schema_map`) was correct all along; only the cached snapshot was stale.

In [5]:
sv = gen.schemaview
print("Before cache_clear(): xsd in namespaces()?", "xsd" in sv.namespaces())
sv.namespaces.cache_clear()
print("After cache_clear():  xsd in namespaces()?", "xsd" in sv.namespaces())

Before cache_clear(): xsd in namespaces()? False
=== namespaces() actually computing (call #2) ===
    schema_map at this moment: ['root', './local_helper', 'childns:child', 'linkml:types']
    xsd already present? True
After cache_clear():  xsd in namespaces()? True


## 5. Why PR #3575's own fix doesn't resolve this case

[PR #3575](https://github.com/linkml/linkml/pull/3575) adds
`linkml.utils.schema_prefix_merge.materialize_prefixes(schemaview)`,
called at the top of `ShaclGenerator.as_graph()`, which copies every
imported schema's own prefixes onto `schemaview.schema.prefixes` --
directly addressing #3574's own "Cause 2" (generators that read
`schema.prefixes` directly only see the root schema's own declarations).
But `ShaclGenerator._add_type()` doesn't read `schema.prefixes` directly
-- it calls `sv.get_uri(rt, expand=True)`, which calls `expand_curie()`,
which calls the *cached* `namespaces()`. Mutating `schema.prefixes`
in-place, without also clearing `namespaces()`'s own cache, is invisible
to that call chain if `namespaces()` was already poisoned earlier --
which, per section 2 above, it already was, before `as_graph()`'s body
even starts.

Reproduced directly: the exact `materialize_prefixes()` function from the
PR, run against this same schema, in the same place the PR itself runs
it.

In [6]:
from linkml_runtime.linkml_model.meta import Prefix


def materialize_prefixes(schemaview):
    """Verbatim copy of PR #3575's own linkml/utils/schema_prefix_merge.py."""
    root = schemaview.schema
    if root.prefixes is None:
        root.prefixes = {}
    schema_map = schemaview.schema_map
    for schema_name in schemaview.imports_closure():
        imported = schema_map.get(schema_name)
        if imported is None or not imported.prefixes:
            continue
        for pfx_name, pfx in imported.prefixes.items():
            if pfx_name in root.prefixes:
                continue
            reference = pfx.prefix_reference
            if reference is None or "://" not in reference:
                continue
            root.prefixes[pfx_name] = Prefix(pfx_name, reference)


# Fresh generator/SchemaView, so namespaces() gets poisoned again exactly
# as it would in a real run (undoing our own cache_clear() above).
gen2 = ShaclGenerator(str(tmpdir / "root.yaml"))
sv2 = gen2.schemaview

materialize_prefixes(sv2)  # the PR's own fix, applied exactly as it runs in as_graph()
print("After materialize_prefixes(): xsd in sv2.schema.prefixes?", "xsd" in sv2.schema.prefixes)
print("After materialize_prefixes(): xsd in sv2.namespaces()?    ", "xsd" in sv2.namespaces())

ttl2 = gen2.serialize()
g2 = Graph()
g2.parse(data=ttl2, format="turtle")
bad2 = {o for s, p, o in g2 if p == SH_DATATYPE and len(str(o)) < 20}
print("Malformed sh:datatype terms after full generation, WITH the PR's own fix applied:", bad2)

=== namespaces() actually computing (call #3) ===


    schema_map at this moment: ['root', './local_helper']
    xsd already present? False
After materialize_prefixes(): xsd in sv2.schema.prefixes? True
After materialize_prefixes(): xsd in sv2.namespaces()?     False
Malformed sh:datatype terms after full generation, WITH the PR's own fix applied: {rdflib.term.URIRef('xsd:string')}


`xsd` is now in `sv2.schema.prefixes` (the PR's fix worked, as far as it
goes) but still absent from `sv2.namespaces()` (the cache is untouched),
and the full generator run still produces the malformed term. The fix
needs one more line -- `schemaview.namespaces.cache_clear()` right after
the mutation -- or, more robustly, `namespaces()` itself needs to stop
being unconditionally cached across the lifetime of a `SchemaView` whose
`schema_map` can still grow.